In [1]:
import time
import requests
import re
from datetime import datetime, timedelta
from dateutil import parser as dateparser
from functools import lru_cache

In [2]:
COINGECKO_BASE = "https://api.coingecko.com/api/v3"


In [3]:
# ---------------------
# Simple caching helpers
# ---------------------
@lru_cache(maxsize=1)
def init_coin_cache():
    """
    Returns a list of coin dicts from CoinGecko /coins/list
    Each dict: { 'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin' }
    Cached in-memory.
    """
    url = f"{COINGECKO_BASE}/coins/list"
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    return r.json()

def _normalize(s: str) -> str:
    return re.sub(r'[^a-z0-9]', '', s.lower())


In [4]:
# ---------------------
# Resolve coin names or symbols to CoinGecko id
# ---------------------
def resolve_coin(query: str):
    """
    Try to resolve common queries into coin id.
    Accepts coin name or symbol, e.g. "btc", "bitcoin", "Ethereum".
    Returns the best-matching coin id (string) or None.
    """
    query_n = _normalize(query)
    if not query_n:
        return None
    coins = init_coin_cache()
    # exact id match
    for c in coins:
        if c['id'] == query_n:
            return c['id']
    # exact symbol or name match
    for c in coins:
        if _normalize(c['symbol']) == query_n or _normalize(c['name']) == query_n:
            return c['id']
    # fuzzy: substring match on name
    for c in coins:
        if query_n in _normalize(c['name']) or query_n in _normalize(c['symbol']):
            return c['id']
    return None

In [5]:

# ---------------------
# Price lookup
# ---------------------
def get_price(coin_ids, vs_currencies=['usd']):
    """
    coin_ids: str or list of coin ids (coingecko IDs, e.g. 'bitcoin') or symbols/names (will resolve).
    vs_currencies: list of currencies like ['usd', 'cad', 'eur']
    Returns: dict like { 'bitcoin': {'usd': 12345.0, 'cad': ...}, 'ethereum': {...} }
    """
    if isinstance(coin_ids, str):
        coin_ids = [coin_ids]
    resolved = []
    for q in coin_ids:
        # if it looks like an id already (no spaces and lowercase), try as is else resolve
        c = resolve_coin(q) or q
        resolved.append(c)
    ids_param = ",".join(resolved)
    vs = ",".join(vs_currencies)
    url = f"{COINGECKO_BASE}/simple/price"
    params = {"ids": ids_param, "vs_currencies": vs}
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    return r.json()

In [6]:
# ---------------------
# Market chart (trend)
# ---------------------
def get_market_chart(coin_id, vs_currency='usd', days=7):
    """
    Returns OHLC-like market_chart data for coin_id over `days` days.
    Data returned by CoinGecko: prices, market_caps, total_volumes
    We'll return the 'prices' list and a small summary (start, end, pct change).
    """
    url = f"{COINGECKO_BASE}/coins/{coin_id}/market_chart"
    params = {"vs_currency": vs_currency, "days": days}
    r = requests.get(url, params=params, timeout=20)
    if r.status_code == 404:
        raise ValueError(f"Coin '{coin_id}' not found on CoinGecko")
    r.raise_for_status()
    j = r.json()
    prices = j.get("prices", [])  # list of [timestamp, price]
    if not prices:
        return {"prices": [], "summary": {}}
    start_price = prices[0][1]
    end_price = prices[-1][1]
    pct_change = (end_price - start_price) / start_price * 100 if start_price else None
    summary = {
        "start_price": start_price,
        "end_price": end_price,
        "pct_change": round(pct_change, 2) if pct_change is not None else None,
        "points": len(prices)
    }
    return {"prices": prices, "summary": summary}


In [7]:
# ---------------------
# Top movers by 24h change
# ---------------------
def get_top_movers(vs_currency='usd', n=10, order='market_cap_desc'):
    """
    Returns the top n coins from /coins/markets with fields:
    id, symbol, name, current_price, price_change_percentage_24h, market_cap
    'order' can be one of CoinGecko's orders (market_cap_desc, market_cap_asc, market_cap_change_24h_desc, etc.)
    """
    url = f"{COINGECKO_BASE}/coins/markets"
    params = {
        "vs_currency": vs_currency,
        "order": order,
        "per_page": n,
        "page": 1,
        "price_change_percentage": "24h"
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    j = r.json()
    simplified = []
    for item in j:
        simplified.append({
            "id": item.get("id"),
            "symbol": item.get("symbol"),
            "name": item.get("name"),
            "current_price": item.get("current_price"),
            "price_change_percentage_24h": item.get("price_change_percentage_24h"),
            "market_cap": item.get("market_cap"),
            "market_cap_rank": item.get("market_cap_rank")
        })
    return simplified

In [8]:
# ---------------------
# Portfolio valuation helper
# ---------------------
def portfolio_value(portfolio, vs_currency='usd'):
    """
    portfolio: list of { 'coin': 'btc'|'bitcoin'|'ethereum', 'amount': float }
    Returns: dict { 'total': float, 'currency': vs_currency, 'breakdown': [...] }
    """
    # build unique coin list
    coins = list({p['coin'] for p in portfolio})
    prices = get_price(coins, vs_currencies=[vs_currency])
    breakdown = []
    total = 0.0
    for p in portfolio:
        coin_q = p['coin']
        amt = float(p.get('amount', 0))
        coin_id = resolve_coin(coin_q) or coin_q
        price_info = prices.get(coin_id) or prices.get(coin_q) or {}
        price = float(price_info.get(vs_currency, 0.0))
        val = amt * price
        breakdown.append({
            "coin": coin_q,
            "coin_id": coin_id,
            "amount": amt,
            "price": price,
            "value": val
        })
        total += val
    return {"total": total, "currency": vs_currency, "breakdown": breakdown}


In [9]:
# ---------------------
# Lightweight intent parsing and handler
# ---------------------
def handle_crypto_request(user_text: str, session: dict = None):
    """
    Tries to infer intent and runs the appropriate helper.
    Returns a dict:
      {
        'ok': True/False,
        'intent': 'price'|'trend'|'top_movers'|'portfolio'|'help',
        'data': ...,
        'message': friendly text (optional)
      }
    session: optional dict to persist small state (not required)
    """
    text = user_text.lower().strip()

    # Intent heuristics
    if re.search(r'\b(top|best|gainers|losers|movers)\b', text):
        # top movers / gainers / losers
        n_match = re.search(r'(\d+)\s*(top|gainers|losers|movers)', text)
        n = int(n_match.group(1)) if n_match else 10
        vs = 'usd'
        vs_m = re.search(r'in\s+([a-z]{3,4})\b', text)
        if vs_m:
            vs = vs_m.group(1)
        movers = get_top_movers(vs_currency=vs, n=n)
        return {"ok": True, "intent": "top_movers", "data": movers, "message": f"Top {n} movers (vs {vs.upper()})"}

    if re.search(r'\bportfolio\b', text):
        # expect syntax like: "portfolio: btc 0.5, eth 2"
        # fallback: ask user for portfolio format
        # try to extract pairs
        pairs = re.findall(r'([a-zA-Z0-9\-\_]+)\s+([0-9]*\.?[0-9]+)', text)
        if not pairs:
            return {"ok": False, "intent": "portfolio", "message": "Please provide your portfolio in the form 'btc 0.5, eth 2'."}
        portfolio = [{"coin": a, "amount": float(b)} for (a, b) in pairs]
        vs = 'usd'
        vs_m = re.search(r'in\s+([a-z]{3,4})\b', text)
        if vs_m:
            vs = vs_m.group(1)
        val = portfolio_value(portfolio, vs_currency=vs)
        return {"ok": True, "intent": "portfolio", "data": val, "message": f"Portfolio value (vs {vs.upper()})"}

    if re.search(r'\b(price|price of|what is the price|quote)\b', text) or re.search(r'\bhow much is\b', text):
        # extract coin names/symbols
        # pattern: words after "price of" or standalone symbols
        coins = re.findall(r'price of ([a-zA-Z0-9\-_,\s]+)', text)
        if coins:
            coin_list = re.split(r'[,\s]+', coins[0].strip())
        else:
            # try to find common tickers / names in the text
            coin_list = re.findall(r'\b(btc|bitcoin|eth|ethereum|ada|doge|dogecoin|xrp|sol|bnb)\b', text)
        if not coin_list:
            # as a last resort try all words as coin queries
            tokens = re.findall(r'[a-zA-Z0-9\-\_]+', text)
            # filter stopwords
            stop = {'price','what','is','the','of','in','usd','cad','eur','show','give','me','please','coin','coins'}
            coin_list = [t for t in tokens if t not in stop][:6]
        vs = 'usd'
        vs_m = re.search(r'in\s+([a-z]{3,4})\b', text)
        if vs_m:
            vs = vs_m.group(1)
        # resolve and fetch
        resolved_ids = []
        for c in coin_list:
            rid = resolve_coin(c) or c
            resolved_ids.append(rid)
        prices = get_price(resolved_ids, vs_currencies=[vs])
        return {"ok": True, "intent": "price", "data": prices, "message": f"Prices (vs {vs.upper()})"}

    if re.search(r'\b(trend|chart|history|performance)\b', text):
        # e.g. "show me bitcoin trend last 7 days"
        d = re.search(r'last\s+(\d+)\s*days', text)
        days = int(d.group(1)) if d else 7
        coin = re.findall(r'[a-zA-Z0-9\-\_]+', text)[0] if re.findall(r'[a-zA-Z0-9\-\_]+', text) else 'bitcoin'
        coin_id = resolve_coin(coin) or coin
        chart = get_market_chart(coin_id, vs_currency='usd', days=days)
        return {"ok": True, "intent": "trend", "coin": coin_id, "days": days, "data": chart, "message": f"{coin_id} trend last {days} days (vs USD)"}

    # fallback: help text
    help_msg = (
        "I can help with crypto prices, trends, top movers, and portfolio valuation.\n"
        "Examples:\n"
        "- 'price of bitcoin in cad'\n"
        "- 'show me btc trend last 30 days'\n"
        "- 'top 5 movers in usd'\n"
        "- 'portfolio: btc 0.5, eth 2 in usd'\n"
    )
    return {"ok": True, "intent": "help", "message": help_msg}

In [10]:
# ---------------------
# Small formatter for pretty plaintext presentation
# ---------------------
def format_response_plain(resp: dict) -> str:
    if not resp.get("ok"):
        return resp.get("message", "Error")

    intent = resp.get("intent")
    if intent == "price":
        data = resp.get("data", {})
        lines = []
        for coin_id, price_map in data.items():
            for cur, val in price_map.items():
                lines.append(f"{coin_id}: {val} {cur.upper()}")
        return "\n".join(lines) if lines else "No price data found."
    if intent == "top_movers":
        items = resp.get("data", [])
        lines = []
        for i, it in enumerate(items, 1):
            lines.append(f"{i}) {it['name']} ({it['symbol'].upper()}) — price: {it['current_price']} — 24h: {it['price_change_percentage_24h']}% — market cap rank: {it.get('market_cap_rank')}")
        return "\n".join(lines)
    if intent == "portfolio":
        data = resp.get("data", {})
        lines = [f"Total: {data['total']:.2f} {data['currency'].upper()}"]
        for b in data['breakdown']:
            lines.append(f"- {b['coin']} ({b['coin_id']}): {b['amount']} × {b['price']:.4f} = {b['value']:.2f} {data['currency'].upper()}")
        return "\n".join(lines)
    if intent == "trend":
        data = resp.get("data", {})
        s = data.get("summary", {})
        return f"{resp.get('coin')} — {s.get('points')} points — start: {s.get('start_price'):.6f} — end: {s.get('end_price'):.6f} — change: {s.get('pct_change')}%"
    if intent == "help":
        return resp.get("message")
    return str(resp)

In [ ]:

# ---------------------
# Example quick-run when module executed directly
# ---------------------
if __name__ == "__main__":
    examples = [
        "What's the price of bitcoin and ethereum in cad?",
        "Show me BTC trend last 14 days",
        "Top 5 movers in usd",
        "Portfolio: btc 0.1, eth 1.5 in usd"
    ]
    for ex in examples:
        try:
            r = handle_crypto_request(ex)
            print(">>>", ex)
            print(format_response_plain(r))
            print("-" * 60)
        except Exception as e:
            print("Error for:", ex, e)

>>> What's the price of bitcoin and ethereum in cad?
No price data found.
------------------------------------------------------------
>>> Show me BTC trend last 14 days
vitanova — 337 points — start: 0.001518 — end: 0.001154 — change: -23.97%
------------------------------------------------------------
>>> Top 5 movers in usd
1) Bitcoin (BTC) — price: 121978 — 24h: 0.52835% — market cap rank: 1
2) Ethereum (ETH) — price: 4469.78 — 24h: -0.46885% — market cap rank: 2
3) XRP (XRP) — price: 2.96 — 24h: -2.79789% — market cap rank: 3
4) Tether (USDT) — price: 1.0 — 24h: -0.02416% — market cap rank: 4
5) BNB (BNB) — price: 1147.67 — 24h: 0.3352% — market cap rank: 5
------------------------------------------------------------
>>> Portfolio: btc 0.1, eth 1.5 in usd
Total: 6720.68 USD
- btc (batcat): 0.1 × 0.0001 = 0.00 USD
- eth (bifrost-bridged-eth-bifrost): 1.5 × 4480.4500 = 6720.67 USD
------------------------------------------------------------


In [13]:

# ---------------------
# Example quick-run when module executed directly
# ---------------------
if __name__ == "__main__":
    examples = [
        "What's the price of bitcoin and ethereum in cad?",
        "Show me BTC trend last 14 days",
        "Top 5 mover coins in usd",
        "Portfolio: btc 0.1, eth 1.5 in usd"
    ]
    for ex in examples:
        try:
            r = handle_crypto_request(ex)
            print(">>>", ex)
            print(format_response_plain(r))
            print("-" * 60)
        except Exception as e:
            print("Error for:", ex, e)

>>> What's the price of bitcoin and ethereum in cad?
No price data found.
------------------------------------------------------------
>>> Show me BTC trend last 14 days
vitanova — 337 points — start: 0.001518 — end: 0.001142 — change: -24.78%
------------------------------------------------------------
>>> Top 5 mover coins in usd
1) Bitcoin (BTC) — price: 121869 — 24h: 0.41652% — market cap rank: 1
2) Ethereum (ETH) — price: 4459.71 — 24h: -0.57202% — market cap rank: 2
3) XRP (XRP) — price: 2.96 — 24h: -3.00217% — market cap rank: 3
4) Tether (USDT) — price: 1.0 — 24h: -0.02405% — market cap rank: 4
5) BNB (BNB) — price: 1146.34 — 24h: 0.21889% — market cap rank: 5
6) Solana (SOL) — price: 226.36 — 24h: -1.9552% — market cap rank: 6
7) USDC (USDC) — price: 0.999709 — 24h: -0.01233% — market cap rank: 7
8) Lido Staked Ether (STETH) — price: 4459.89 — 24h: -0.50369% — market cap rank: 8
9) Dogecoin (DOGE) — price: 0.24873 — 24h: -3.78574% — market cap rank: 9
10) TRON (TRX) — price: 0

In [14]:

# ---------------------
# Example quick-run when module executed directly
# ---------------------
if __name__ == "__main__":
    examples = [
        "What's the price of bitcoin?",
        "price of ETH in cad?",
        "Show me BTC and LTC prices in eur",
        "Show me BTC trend last 14 days",
        "Top 5 mover coins in usd",
        "Portfolio: btc 0.1, eth 1.5 in usd"
    ]
    for ex in examples:
        try:
            r = handle_crypto_request(ex)
            print(">>>", ex)
            print(format_response_plain(r))
            print("-" * 60)
        except Exception as e:
            print("Error for:", ex, e)

>>> What's the price of bitcoin?
bitcoin: 121835 USD
------------------------------------------------------------
>>> price of ETH in cad?
bifrost-bridged-eth-bifrost: 6258.17 CAD
incoin-2: 0.0004785 CAD
------------------------------------------------------------
>>> Show me BTC and LTC prices in eur
I can help with crypto prices, trends, top movers, and portfolio valuation.
Examples:
- 'price of bitcoin in cad'
- 'show me btc trend last 30 days'
- 'top 5 movers in usd'
- 'portfolio: btc 0.5, eth 2 in usd'

------------------------------------------------------------
>>> Show me BTC trend last 14 days
vitanova — 337 points — start: 0.001518 — end: 0.001142 — change: -24.78%
------------------------------------------------------------
>>> Top 5 mover coins in usd
1) Bitcoin (BTC) — price: 121833 — 24h: 0.38663% — market cap rank: 1
2) Ethereum (ETH) — price: 4457.71 — 24h: -0.61683% — market cap rank: 2
3) XRP (XRP) — price: 2.95 — 24h: -3.15198% — market cap rank: 3
4) Tether (USDT) —